# Manhattan Tiny OCR Verification

Visual verification of OCR detections from the Manhattan 8K tiny test run.
Each image is shown with detected text bounding boxes overlaid to identify potential hallucinations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
from IPython.display import display, HTML

# Configuration
PARQUET_PATH = "/share/ju/matt/mllmsci/outputs/ocr_manhattan_tiny/detections.parquet"

# Load data
df = pd.read_parquet(PARQUET_PATH)
print(f"Total detection rows: {len(df)}")
print(f"Unique images: {df['image_path'].nunique()}")
print(f"Columns: {df.columns.tolist()}")


## Summary Statistics


In [ ]:
# Quick summary
print("=== DETECTION SUMMARY ===")
detections_per_image = df[df['text'].notna()].groupby('image_path').size()
print(f"Total images: {df['image_path'].nunique()}")
print(f"Total detections (with text): {df['text'].notna().sum()}")
print(f"Detections per image: min={detections_per_image.min()}, max={detections_per_image.max()}, mean={detections_per_image.mean():.1f}")

# Confidence distribution
if 'confidence' in df.columns:
    print(f"\n=== CONFIDENCE DISTRIBUTION ===")
    print(df['confidence'].value_counts())

# Text type distribution
if 'text_type' in df.columns:
    print(f"\n=== TEXT TYPE DISTRIBUTION ===")
    print(df['text_type'].value_counts())

# All detected texts
print(f"\n=== ALL DETECTED TEXTS ({df['text'].notna().sum()} total) ===")
for _, row in df[df['text'].notna()].iterrows():
    face = Path(row['image_path']).stem
    conf = row.get('confidence', '?')
    print(f"  [{face}] [{conf}] '{row['text']}'")


## Visual Verification

Each image is shown with detected bounding boxes. **Red boxes indicate potential hallucinations** if you can't find the text in that location.


In [ ]:
def visualize_detections_detailed(image_path, detections_df, figsize=(20, 20)):
    """Visualize OCR detections on an image with detailed labels."""
    
    # Load image
    img = Image.open(image_path)
    img_width, img_height = img.size
    
    # Get coordinate range from data
    if len(detections_df) > 0 and 'bbox_max_x' in detections_df.columns:
        bbox_max_x = detections_df['bbox_max_x'].iloc[0]
        bbox_max_y = detections_df['bbox_max_y'].iloc[0]
    else:
        bbox_max_x = bbox_max_y = 999
    
    # Filter to actual detections
    dets = detections_df[detections_df['text'].notna()]
    
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)
    
    # Color map by confidence
    conf_colors = {
        'perfect': 'lime',
        'very_high': 'green',
        'high': 'yellow',
        'moderate': 'orange',
        'low': 'red',
        'very_low': 'darkred',
    }
    
    print(f"\n{'='*60}")
    print(f"IMAGE: {Path(image_path).name} ({img_width}x{img_height})")
    print(f"DETECTIONS: {len(dets)}")
    print(f"{'='*60}")
    
    for i, (_, row) in enumerate(dets.iterrows()):
        # Skip if no bbox
        if pd.isna(row.get('bbox_x1')):
            continue
            
        # Convert global coords to pixel coords
        x1 = row['bbox_x1'] / bbox_max_x * img_width
        y1 = row['bbox_y1'] / bbox_max_y * img_height
        x2 = row['bbox_x2'] / bbox_max_x * img_width
        y2 = row['bbox_y2'] / bbox_max_y * img_height
        
        width = x2 - x1
        height = y2 - y1
        
        # Get color based on confidence
        conf = row.get('confidence', 'moderate')
        color = conf_colors.get(conf, 'red')
        
        # Draw rectangle
        rect = patches.Rectangle((x1, y1), width, height, 
                                   linewidth=3, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Add number label
        ax.text(x1, y1-10, str(i+1), fontsize=14, color=color, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        
        # Print detection details
        text_type = row.get('text_type', '?')
        print(f"  [{i+1}] [{conf}] [{text_type}] '{row['text']}'")
        print(f"      bbox: ({row['bbox_x1']:.0f}, {row['bbox_y1']:.0f}) -> ({row['bbox_x2']:.0f}, {row['bbox_y2']:.0f})")
        print(f"      pixel: ({x1:.0f}, {y1:.0f}) -> ({x2:.0f}, {y2:.0f})")
    
    ax.set_title(f"{Path(image_path).name} - {len(dets)} detections", fontsize=16)
    ax.axis('off')
    
    # Legend
    legend_patches = [patches.Patch(color=c, label=l) for l, c in conf_colors.items()]
    ax.legend(handles=legend_patches, loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    return dets


## Verify Each Face

Running through all 4 faces (B, F, L, R) to visually check detections.


In [ ]:
# Visualize all faces
unique_images = df['image_path'].unique()
print(f"Processing {len(unique_images)} images...")

for img_path in sorted(unique_images):
    img_df = df[df['image_path'] == img_path]
    visualize_detections_detailed(img_path, img_df, figsize=(18, 18))


## Coordinate System Analysis

The 8192x8192 images are tiled to 1024x1024. Check the coordinate system being used.


In [ ]:
# Analyze coordinate system
print("=== COORDINATE SYSTEM ===")
print(f"bbox_max_x: {df['bbox_max_x'].unique()}")
print(f"bbox_max_y: {df['bbox_max_y'].unique()}")
print(f"n_tiles_x: {df['n_tiles_x'].unique()}")
print(f"n_tiles_y: {df['n_tiles_y'].unique()}")

# Bbox ranges
dets = df[df['text'].notna()]
if len(dets) > 0:
    print(f"\n=== BBOX RANGES (actual detections) ===")
    print(f"bbox_x1: min={dets['bbox_x1'].min():.0f}, max={dets['bbox_x1'].max():.0f}")
    print(f"bbox_y1: min={dets['bbox_y1'].min():.0f}, max={dets['bbox_y1'].max():.0f}")
    print(f"bbox_x2: min={dets['bbox_x2'].min():.0f}, max={dets['bbox_x2'].max():.0f}")
    print(f"bbox_y2: min={dets['bbox_y2'].min():.0f}, max={dets['bbox_y2'].max():.0f}")
    
    # Expected for 8x8 tiles with 0-999 per tile: max should be ~7999
    expected_max = (8 * 999)
    print(f"\nExpected max for 8x8 tiles: {expected_max}")
else:
    print("No detections with text found!")


## Hallucination Checklist

Mark potential hallucinations by comparing the numbered boxes in the images above with the printed detection list.

**Signs of hallucination:**
- Text reported in areas with no visible text
- Text content doesn't match what's visible
- Bounding box in wrong location
- Very generic text that appears repeatedly ("STOP", single numbers, etc.)
